<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Small, lasting shifts: CUSUM and EWMA

**[Small, lasting shifts: CUSUM and EWMA](https://learning.nextia-ai.com/courses/anomaly-detection/m03/small-lasting-shifts-cusum-and-ewma/)** · Anomaly Detection: Find What Is Unusual · Module 3, lesson 4 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build a CUSUM chart and an EWMA chart, which add up small differences over days, to choose their settings from the shift you must catch, and to compare their average run lengths with the Shewhart charts on the same data.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (less than 1 MB). No account. |
| **You should know** | The average run length, from [Rules for patterns, and the price of each rule](https://learning.nextia-ai.com/courses/anomaly-detection/m03/rules-for-patterns/), and the Laney chart, from [Charts for rates and counts](https://learning.nextia-ai.com/courses/anomaly-detection/m03/charts-for-rates-and-counts/). |
| **You do not need** | A statistics package: each chart is a loop of a few lines. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m03/small-lasting-shifts-cusum-and-ewma/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M03-L04-small-lasting-shifts-cusum-and-ewma/small-lasting-shifts-cusum-and-ewma-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`refunds_daily.csv`, `refunds_agents.csv`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "refunds_daily.csv": "04c6384d1fbe8da908a739edc084eabc413581d75fce7cd84dcf7a90988996a6",
    "refunds_agents.csv": "1a883275f23988a40d99955e17f23ba132d98b7fb67912b0e197478cfff21782",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. CUSUM on eleven days

A Shewhart chart looks at one day. A small, lasting shift moves each day only a little, so no single day crosses the limit. The **CUSUM** (cumulative sum) chart adds up the part of each day that is above a small **slack** k, and alarms when the total passes a level h. A day below the slack takes some of the total away, but never below 0.

> **Predict.** Eleven days have z values from −0.5 to 1.4: none is near 3. With k = 0.5 and h = 4, does the CUSUM alarm, and on which day?

> **Check.** You should see the first `True` on day 11.

In [ ]:
tiny = pd.DataFrame({"z": [0.2, -0.5, 0.9, 1.1, 0.4, 1.3, 0.8, 1.2, 0.6, 1.4, 1.0]},
                    index=pd.RangeIndex(1, 12, name="day"))      # 11 days, in sigmas from the centre
k, h = 0.5, 4.0                                       # k: the slack per day; h: the alarm level
total, sums = 0.0, []
for v in tiny.z:
    total = max(0.0, total + v - k)                   # add what is above the slack; never below 0
    sums.append(round(total, 2))
tiny["z - k"] = tiny.z - k
tiny["CUSUM"] = sums
tiny["alarm"] = tiny.CUSUM > h
print(tiny.to_string())

## 2. EWMA on the same days

The **EWMA** (exponentially weighted moving average) chart keeps a running average: each day, λ (lambda) of the new value and 1 − λ of the old average. With λ = 0.1, today counts 10% and the past 90%. The average moves less than one day does, so its limits are narrower.

Run the next cell. It prints the EWMA, its limit (L = 2.6 sigmas of the average) and the alarm for each day.

> **Check.** You should see the first `True` on day 11.

In [ ]:
lam, L = 0.1, 2.6
e, days = 0.0, []
for t, v in enumerate(tiny.z, start=1):
    e = lam * v + (1 - lam) * e                        # 10% of today, 90% of the past
    limit = L * np.sqrt(lam / (2 - lam) * (1 - (1 - lam) ** (2 * t)))
    days.append({"day": t, "z": v, "EWMA": round(e, 3), "limit": round(limit, 3), "alarm": e > limit})
print(pd.DataFrame(days).set_index("day").to_string())

## 3. Settings from the shift you must catch

Now Larkfield's refunds. The CUSUM and the EWMA use z on the Laney scale from the last lesson. The slack k is half the shift that we must catch, in sigmas of one day.

> **Check.** You should see a shift of 0.482 sigma and k = 0.241.

In [ ]:
import compare
daily = pd.read_csv("data/refunds_daily.csv", parse_dates=["date"]).set_index("date")
x = daily.dropna()
p1 = x.loc["2024-04-01":"2024-12-31"]
pbar = p1.refunds.sum() / p1.orders.sum()
z = (x.refund_rate - pbar) / np.sqrt(pbar * (1 - pbar) / x.orders)    # z of the p-chart
laney = z.loc[:"2024-12-31"].diff().abs().mean() / 1.128              # sigma_z of the Laney chart
zl = z / laney                                                       # z on the Laney scale
shift = 0.003 / (np.sqrt(pbar * (1 - pbar) / p1.orders.mean()) * laney)   # 0.3 points, in sigmas of one day
k = shift / 2
print(f"Laney sigma_z {laney:.3f}; the 0.3-point shift is {shift:.3f} sigma; k = {k:.3f}")

The planted shift is less than half a sigma a day. That is why the charts of the last lessons took months.

Run the next cell. It defines `cusum()` and `ewma()` for a series of days. It prints nothing.

In [ ]:
def cusum(zs, k, h, side="both"):
    """Tabular CUSUM: True on each alarm day. It starts again from 0 after an alarm."""
    hi = lo = 0.0
    out = []
    for v in zs:
        hi, lo = max(0.0, hi + v - k), max(0.0, lo - v - k)
        alarm = hi > h or (side == "both" and lo > h)
        out.append(alarm)
        if alarm:
            hi = lo = 0.0
    return pd.Series(out, index=zs.index)

def ewma(zs, lam, L, side="both"):
    """EWMA chart: True on each alarm day. It starts again from 0 after an alarm."""
    e, out = 0.0, []
    for t, v in enumerate(zs, start=1):
        e = lam * v + (1 - lam) * e
        limit = L * np.sqrt(lam / (2 - lam) * (1 - (1 - lam) ** (2 * t)))
        alarm = e > limit if side == "high" else abs(e) > limit
        out.append(alarm)
        if alarm:
            e = 0.0
    return pd.Series(out, index=zs.index)

## 4. Choose h by simulation

A larger h means fewer false alarms and later real alarms. We choose h so that a false alarm comes about once a year: an in-control ARL near 370 days, as a 3-sigma Shewhart chart on textbook data.

Run the next cell. It simulates 2,000 runs of 1,500 days without a shift and 2,000 with the 0.3-point shift, as in the last two lessons. It takes a few seconds.

In [ ]:
orders_pool = p1.orders.to_numpy()
center_i = p1.refund_rate.mean()
sigma_i = p1.refund_rate.diff().abs().mean() / 1.128

def simulate(shift_points, runs=2000, days=1500):
    """runs x days of daily refund rates as Larkfield's process makes them; z for each kind of chart."""
    n = rng.choice(orders_pool, size=(runs, days))
    p = (pbar + shift_points / 100) * np.exp(rng.normal(0, 0.10, (runs, days)))
    r = rng.binomial(n.astype(int), np.clip(p, 0, 1)) / n
    zp = (r - pbar) / np.sqrt(pbar * (1 - pbar) / n)
    return {"I-MR": (r - center_i) / sigma_i, "p": zp, "Laney": zp / laney}

rng = np.random.default_rng(34)
normal, shifted = simulate(0.0), simulate(0.3)        # 2,000 runs of 1,500 days each: no shift, then the shift
print("simulated days:", 2 * 2000 * 1500)

In [ ]:
def cusum_2d(zs, k, h):
    hi, lo = np.zeros(zs.shape[0]), np.zeros(zs.shape[0])
    out = np.zeros(zs.shape, bool)
    for t in range(zs.shape[1]):
        hi = np.maximum(0, hi + zs[:, t] - k)
        lo = np.maximum(0, lo - zs[:, t] - k)
        out[:, t] = (hi > h) | (lo > h)
    return out

def ewma_2d(zs, lam, L):
    e = np.zeros(zs.shape[0])
    out = np.zeros(zs.shape, bool)
    for t in range(zs.shape[1]):
        e = lam * zs[:, t] + (1 - lam) * e
        out[:, t] = np.abs(e) > L * np.sqrt(lam / (2 - lam) * (1 - (1 - lam) ** (2 * (t + 1))))
    return out

def arl(alarms):
    """Average run length: the mean number of days to the first alarm (1,501 if a run has none)."""
    return float(np.where(alarms.any(axis=1), alarms.argmax(axis=1) + 1, alarms.shape[1] + 1).mean())

for h in (4.0, 6.0, 8.0):
    print(f"CUSUM with h = {h}: a false alarm every {arl(cusum_2d(normal['Laney'], k, h)):.1f} days")

> **Check.** You should see 427.7 days for h = 8.0.

h = 8 is the value nearest to 370 in the grid of the measurement (3.0 to 8.0). For the EWMA, λ = 0.1 and L = 2.6 were chosen the same way.

## 5. Five charts on the same shift

> **Predict.** Which chart finds the 0.3-point shift first on Larkfield's data? And on average over 2,000 simulated runs? Write both answers, then run the cell. It takes about half a minute.

In [ ]:
h, lam, L = 8.0, 0.1, 2.6
charts = {
    "I-MR": (lambda zz: np.abs(zz["I-MR"]) > 3, (x.refund_rate - center_i) / sigma_i > 3),
    "p-chart": (lambda zz: np.abs(zz["p"]) > 3, z > 3),
    "Laney p'": (lambda zz: np.abs(zz["Laney"]) > 3, zl > 3),
    "CUSUM": (lambda zz: cusum_2d(zz["Laney"], k, h), cusum(zl, k, h, side="high")),
    "EWMA": (lambda zz: ewma_2d(zz["Laney"], lam, L), ewma(zl, lam, L, side="high")),
}
rows = []
for name, (sim_alarms, real_high) in charts.items():
    report = compare.alarm_report(x.index, real_high & (x.index >= "2025-01-01"), "2025-04-07", ("2025-01-01", "2025-04-06"))
    rows.append({"chart": name, "first alarm": report["first alarm"], "days after the shift": report["days to alarm"],
                 "false alarms a month (stable months)": report["false alarms per month"],
                 "days to a false alarm (simulated)": round(arl(sim_alarms(normal)), 1),
                 "days to catch 0.3 points (simulated)": round(arl(sim_alarms(shifted)), 1)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))

> **Check.** You should see `2025-05-12` for the EWMA and `2025-05-20` for the CUSUM.

On the real data, the EWMA alarmed 35 days after the shift and the CUSUM 43 days after it, against 98 days for the Shewhart charts. None gave a false alarm in the stable months. In the simulation, the EWMA caught the shift in 22.4 days on average and the CUSUM in 28.5, and both gave fewer false alarms than the Shewhart charts.

## 6. One agent: the honest limit

The shift above is Larkfield-wide: about 680 orders a day. One agent has far fewer. Run the next cell. It runs a per-agent p-chart and CUSUM (k = 0.5, h = 4) and prints agent A-17, whose rate we planted 3 points higher from 2025-07-07.

In [ ]:
agents = pd.read_csv("data/refunds_agents.csv", parse_dates=["date"])
agents = agents[agents.orders.notna() & (agents.orders > 0) & (agents.date != "2025-11-10")]   # no broken-feed day
a1 = agents[agents.date <= "2024-12-31"]
p_agent = a1.refunds.sum() / a1.orders.sum()
first = {}
for agent, g in agents[agents.date >= "2025-01-01"].groupby("agent_id"):
    g = g.set_index("date")
    za = (g.refunds / g.orders - p_agent) / np.sqrt(p_agent * (1 - p_agent) / g.orders)
    c = cusum(za, 0.5, 4.0)
    first[agent] = {"CUSUM": next((str(d.date()) for d in c.index[c & (c.index >= "2025-07-07")]), None),
                    "p-chart": next((str(d.date()) for d in za.index[(za.abs() > 3) & (za.index >= "2025-07-07")]), None)}
print("A-17:", first["A-17"])
print(f"orders per agent and day: {agents.orders.mean():.1f}")

> **Check.** You should see `2025-07-13` for both charts.

A big shift on one agent is easy: 6 days. Now a 0.3-point shift on one agent with about 25 orders a day. Run the next cell. It uses the same random numbers in the same order as the course's measurement, so it gives the same averages.

In [ ]:
n_agent = int(round(agents.orders.mean()))
sd_agent = np.sqrt(p_agent * (1 - p_agent) / n_agent)
z0 = (rng.binomial(n_agent, p_agent, (2000, 1500)) / n_agent - p_agent) / sd_agent          # one agent, no shift
z1 = (rng.binomial(n_agent, p_agent + 0.003, (2000, 1500)) / n_agent - p_agent) / sd_agent  # +0.3 points
k_agent = (0.003 / sd_agent) / 2
print(f"for one agent, 0.3 points is {0.003 / sd_agent:.3f} sigma")
print(f"CUSUM h = 16: a false alarm every {arl(cusum_2d(z0, k_agent, 16.0)):.1f} days, "
      f"the shift caught after {arl(cusum_2d(z1, k_agent, 16.0)):.1f} days")

> **Check.** You should see about 290 days to a false alarm and 160.9 days to catch the shift.

For one agent, 0.3 points is only about a tenth of a sigma a day. Even a CUSUM tuned for it needs more than five months on average. Size of the shift against size of the sample: no chart can see a change smaller than the noise of the data it watches.

## 7. Check yourself

**Your turn 1.** The CUSUM's k is half the shift to catch, in sigmas of one day. What is k for a shift of 0.5 points? Use `pbar`, `p1` and `laney`, round to 3 decimals, and write it as `k_half_point`.

In [ ]:
k_half_point = round((0.005 / (np.sqrt(pbar * (1 - pbar) / p1.orders.mean()) * laney)) / 2, 3)

In [ ]:
expect('k_half_point', k_half_point, 0.402, tolerance=0.001)

**Your turn 2.** Change one thing: run `ewma(zl, 0.2, 2.6, side="high")`, with λ = 0.2. On which day after the shift (from 2025-04-07) is its first alarm? Write it as `"YYYY-MM-DD"` in `ewma_02_day`.

In [ ]:
e2 = ewma(zl, 0.2, 2.6, side="high")
ewma_02_day = str(e2.index[e2 & (e2.index >= "2025-04-07")][0].date())

In [ ]:
expect_hash('ewma_02_day', ewma_02_day, '3278dd541e44b9f9')

**Your turn 3.** How many days after 2025-07-07 did the CUSUM find A-17? Write the number as `a17_days`.

In [ ]:
a17_days = (pd.Timestamp(first["A-17"]["CUSUM"]) - pd.Timestamp("2025-07-07")).days

In [ ]:
expect('a17_days', a17_days, 6)

## Recap

- CUSUM adds up the part of each day above a slack k; EWMA keeps a running average. Both see a small, lasting shift that no single day shows.
- Choose k from the shift you must catch, and h (or L) from the false alarms you can accept, by simulation.
- On the 0.3-point shift: EWMA 22.4 days and CUSUM 28.5 days on average, against 60.6 for the I-MR chart. For one agent, the same shift needs 160.9 days.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Has the distribution changed?](https://learning.nextia-ai.com/courses/anomaly-detection/m03/has-the-distribution-changed/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m03/small-lasting-shifts-cusum-and-ewma/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.